# Single Pendulum: Deep Q-Network

Complete value-based DQN using JAX, Flax, Optax, replay memory, and a target network.
Run a smoke test before increasing the step budget. All settings likely to need tuning are
grouped in one cell; infrastructure and full-state resume are already implemented.


## 1. Runtime setup
This notebook is self-contained and does not clone TIPy.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_COLAB or IN_KAGGLE:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--upgrade", "-q",
        "jax[cuda12]", "mujoco>=3.2", "gymnasium>=1.0", "flax==0.12.8", "optax==0.2.8",
        "pandas>=2.0", "matplotlib>=3.8", "imageio>=2.34", "imageio-ffmpeg>=0.5",
    ], check=True)

# Keep learning arrays resident on the accelerator and use EGL for GPU-backed video rendering.
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "true")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")

import flax
import jax
import jax.numpy as jnp
import optax

gpu_devices = [device for device in jax.devices() if device.platform == "gpu"]
print(f"JAX {jax.__version__} | Flax {flax.__version__} | Optax {optax.__version__}")
print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())
if (IN_COLAB or IN_KAGGLE) and not gpu_devices:
    raise RuntimeError(
        "GPU accelerator required: enable a GPU in the Colab or Kaggle notebook settings, "
        "restart the runtime, and run all cells again."
    )
if (IN_COLAB or IN_KAGGLE) and flax.__version__ != "0.12.8":
    raise RuntimeError("Restart the notebook runtime, then run all cells again.")

accelerator_probe = jax.jit(lambda x: x @ x)(jnp.ones((64, 64), dtype=jnp.float32))
accelerator_probe.block_until_ready()
probe_device = next(iter(accelerator_probe.devices()))
if (IN_COLAB or IN_KAGGLE) and probe_device.platform != "gpu":
    raise RuntimeError("GPU accelerator required, but the JIT probe ran on " + str(probe_device))
print("JIT accelerator probe:", probe_device)


## 2. Tuning and persistent paths


In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/ProjectsRuns/TIPy/runs/single/dqn/run-001") if IN_COLAB else Path.cwd() / "dqn-run"
CHECKPOINT_DIR, METRICS_PATH = OUTPUT_DIR / "checkpoints", OUTPUT_DIR / "metrics.csv"
DASHBOARD_PATH, TITLE = OUTPUT_DIR / "dashboard.png", "TIPy Single Pendulum DQN"
SEED, TOTAL_STEPS, MAX_EPISODE_STEPS, ACTION_LIMIT = 42, 150_000, 2000, 100.0
LEARNING_RATE, GAMMA, BATCH_SIZE = 3e-4, 0.99, 256
BUFFER_CAPACITY, WARMUP_STEPS, TRAIN_FREQUENCY, TARGET_FREQUENCY = 50_000, 1_000, 4, 500
EPSILON_START, EPSILON_END, EPSILON_DECAY_STEPS = 1.0, 0.05, int(TOTAL_STEPS * .7)
CHECKPOINT_EVERY, SMOKE_TEST = 25, False
if SMOKE_TEST:
    TOTAL_STEPS, MAX_EPISODE_STEPS, WARMUP_STEPS, CHECKPOINT_EVERY = 500, 100, 64, 1
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Plant and environment
Rail failure terminates; time limits truncate and retain Bellman bootstrapping.


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np

MODEL_XML = r"""
<mujoco model="cartpole_single">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4" />
  <default>
    <geom contype="0" conaffinity="0" />
  </default>
  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>
  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole_mat" rgba="0.2 0.75 0.3 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>
  <worldbody>
    <camera name="spectate" pos="0 3 1.4" fovy="90" xyaxes="-1 0 0 0 -0.1240 0.9923" />
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />
    <body name="frame">
      <geom name="rail" type="box" pos="0 0 0.85" size="2.2 0.05 0.05" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-2.2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="2.2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
    </body>
    <body name="cart" pos="0 0 1">
      <joint name="cart_slide" type="slide" axis="1 0 0" range="-2.2 2.2" frictionloss="0.02" damping="0.1" />
      <inertial pos="0 0 0" mass="2" diaginertia="0.0333 0.0333 0.0333" />
      <geom name="cart_geom" type="box" size="0.125 0.08 0.1" mass="2.0" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.095 0" axisangle="1 0 0 1.5708" size="0.015 0.03" material="metal_mat" />
      <body name="pole" pos="0 0.11 0" quat="6.12323399574e-17 0 -1 0">
        <joint name="pole_hinge" type="hinge" axis="0 -1 0" frictionloss="0.01" damping="0.03" ref="3.1415926535897931" limited="false" />
        <inertial pos="0 0 0.3" mass="0.7" diaginertia="0.0210233333333 0.0210933333333 0.000116666666667" />
        <site name="pole_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole_geom" type="box" pos="0 0 0.3" size="0.02 0.01 0.3" mass="0.7" material="pole_mat" />
        <site name="pole_tip_site" pos="0 0 0.6" size="0.015" type="sphere" material="site_mat" />
      </body>
    </body>
    <camera name="replay" pos="0 6 1.4" fovy="50" xyaxes="-1 0 0 0 -0.15 0.988686" />
  </worldbody>
  <sensor>
    <jointpos name="cart_position" joint="cart_slide" />
    <jointpos name="pole1_relative_angle" joint="pole_hinge" />
    <jointvel name="cart_velocity" joint="cart_slide" />
    <jointvel name="pole1_relative_velocity" joint="pole_hinge" />
  </sensor>
  <actuator>
    <motor name="cart_motor" joint="cart_slide" gear="1" ctrlrange="-100 100" forcerange="-100 100" />
  </actuator>
</mujoco>
"""

def normalize_angle(theta):
    return float((theta + np.pi) % (2.0 * np.pi) - np.pi)

class SinglePendulumEnv(gym.Env):
    """Self-contained MuJoCo cart-pole swing-up environment."""
    def __init__(self, discrete=True, max_episode_steps=2000, action_limit=100.0):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_string(MODEL_XML)
        self.data = mujoco.MjData(self.model)
        self.discrete = discrete
        self.max_episode_steps = max_episode_steps
        self.action_limit = float(action_limit)
        self.rail_limit = 2.2
        self.dt = float(self.model.opt.timestep)
        self.current_step = 0
        self.observation_space = spaces.Box(-1.0, 1.0, shape=(5,), dtype=np.float32)
        if discrete:
            self.action_table = np.linspace(-self.action_limit, self.action_limit, 7)
            self.action_space = spaces.Discrete(7)
        else:
            self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)

    def _observation(self):
        x, theta = map(float, self.data.qpos)
        dx, dtheta = map(float, self.data.qvel)
        theta = normalize_angle(theta)
        return np.array([
            np.clip(x / self.rail_limit, -1.0, 1.0),
            np.clip(dx / 5.0, -1.0, 1.0),
            np.cos(theta), np.sin(theta),
            np.clip(dtheta / 15.0, -1.0, 1.0),
        ], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] = [self.np_random.uniform(-0.02, 0.02),
                             np.pi + self.np_random.uniform(-0.05, 0.05)]
        self.data.qvel[:] = self.np_random.uniform([-0.01, -0.02], [0.01, 0.02])
        mujoco.mj_forward(self.model, self.data)
        return self._observation(), {}

    def step(self, action):
        self.current_step += 1
        if self.discrete:
            force = float(self.action_table[int(action)])
        else:
            force = float(np.asarray(action).reshape(-1)[0]) * self.action_limit
        force = float(np.clip(force, -self.action_limit, self.action_limit))
        self.data.ctrl[0] = force
        mujoco.mj_step(self.model, self.data)
        x, theta = map(float, self.data.qpos)
        dx, dtheta = map(float, self.data.qvel)
        theta = normalize_angle(theta)
        reward = (np.cos(theta) - 0.2 * (x / self.rail_limit) ** 2
                  - 0.01 * dx ** 2 - 0.005 * dtheta ** 2
                  - 0.001 * (force / self.action_limit) ** 2
                  + (2.0 if abs(theta) < 0.35 else 0.0))
        terminated = bool(abs(x) >= self.rail_limit)
        if terminated:
            reward -= 10.0
        truncated = bool(self.current_step >= self.max_episode_steps)
        info = {"x": x, "theta": theta, "dx": dx, "dtheta": dtheta, "force": force}
        return self._observation(), float(reward), terminated, truncated, info

# Contract and reproducibility checks.
_env = SinglePendulumEnv(max_episode_steps=10)
_a, _ = _env.reset(seed=7)
_b, _ = _env.reset(seed=7)
assert _a.shape == (5,) and np.allclose(_a, _b)
assert _env.action_space.n == 7 and np.all(np.isfinite(_a))
print("Environment check passed; initial observation:", _a)


## 4. Replay memory
Preallocated CPU arrays move only sampled batches to the accelerator.


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim, rng):
        self.capacity, self.obs_dim, self.rng = capacity, obs_dim, rng
        self.obs = np.zeros((capacity, obs_dim), np.float32); self.next_obs = np.zeros_like(self.obs)
        self.actions = np.zeros(capacity, np.int32); self.rewards = np.zeros(capacity, np.float32)
        self.terminals = np.zeros(capacity, np.float32); self.pointer = self.size = 0

    def add(self, obs, action, reward, next_obs, terminal):
        i = self.pointer; self.obs[i] = obs; self.next_obs[i] = next_obs
        self.actions[i] = action; self.rewards[i] = reward; self.terminals[i] = terminal
        self.pointer = (i + 1) % self.capacity; self.size = min(self.size + 1, self.capacity)

    def sample(self, count):
        i = self.rng.integers(self.size, size=count)
        return {"obs": self.obs[i], "next_obs": self.next_obs[i], "actions": self.actions[i],
                "rewards": self.rewards[i], "terminals": self.terminals[i]}

    def state_dict(self):
        return {"capacity": self.capacity, "obs_dim": self.obs_dim, "pointer": self.pointer,
                "size": self.size, "obs": self.obs, "next_obs": self.next_obs,
                "actions": self.actions, "rewards": self.rewards, "terminals": self.terminals}

    def load_state_dict(self, state):
        if (state["capacity"], state["obs_dim"]) != (self.capacity, self.obs_dim): raise ValueError("buffer mismatch")
        self.pointer, self.size = state["pointer"], state["size"]
        for name in ("obs", "next_obs", "actions", "rewards", "terminals"):
            getattr(self, name)[:] = state[name]


## 5. Q-network
The output has one unrestricted return estimate per discrete action.


In [ ]:
import flax.linen as nn
import jax.numpy as jnp
import optax

class QNetwork(nn.Module):
    actions: int
    @nn.compact
    def __call__(self, x):
        x = nn.relu(nn.Dense(128)(x)); x = nn.relu(nn.Dense(128)(x))
        return nn.Dense(self.actions)(x)


## 6. DQN agent
Only true terminals remove the target-network bootstrap term.


In [ ]:
class DQNAgent:
    def __init__(self, obs_dim, actions, seed):
        self.actions, self.rng, self.total_steps = actions, np.random.default_rng(seed), 0
        self.epsilon = EPSILON_START; self.net = QNetwork(actions)
        self.online = self.net.init(jax.random.PRNGKey(seed), jnp.zeros((1, obs_dim)))
        self.target = self.online
        self.optimizer = optax.chain(optax.clip_by_global_norm(10), optax.adam(LEARNING_RATE))
        self.opt_state = self.optimizer.init(self.online)
        self.buffer = ReplayBuffer(BUFFER_CAPACITY, obs_dim, self.rng)
        self._predict = jax.jit(lambda params, x: self.net.apply(params, x))
        self._update = jax.jit(self._update_impl)

    def _update_impl(self, online, target, opt_state, batch):
        def loss_fn(params):
            q = self.net.apply(params, batch["obs"])
            selected = q[jnp.arange(batch["actions"].shape[0]), batch["actions"]]
            next_q = jnp.max(self.net.apply(target, batch["next_obs"]), axis=-1)
            targets = batch["rewards"] + GAMMA * next_q * (1 - batch["terminals"])
            return jnp.mean((selected - jax.lax.stop_gradient(targets)) ** 2)
        loss, grads = jax.value_and_grad(loss_fn)(online)
        updates, opt_state = self.optimizer.update(grads, opt_state, online)
        return optax.apply_updates(online, updates), opt_state, loss

    def act(self, obs, greedy=False):
        if not greedy and self.rng.random() < self.epsilon: return int(self.rng.integers(self.actions))
        q = np.asarray(self._predict(self.online, jnp.asarray(obs)[None]))[0]
        return int(self.rng.choice(np.flatnonzero(q == q.max())))

    def observe(self, obs, action, reward, next_obs, terminal):
        self.buffer.add(obs, action, reward, next_obs, terminal); self.total_steps += 1
        fraction = min(1.0, self.total_steps / max(1, EPSILON_DECAY_STEPS))
        self.epsilon = EPSILON_START + fraction * (EPSILON_END - EPSILON_START)
        loss = None
        if self.total_steps >= WARMUP_STEPS and self.total_steps % TRAIN_FREQUENCY == 0 and self.buffer.size >= BATCH_SIZE:
            batch = jax.tree.map(jnp.asarray, self.buffer.sample(BATCH_SIZE))
            self.online, self.opt_state, loss = self._update(self.online, self.target, self.opt_state, batch)
            loss = float(loss)
        if self.total_steps % TARGET_FREQUENCY == 0: self.target = self.online
        return loss

    def state_dict(self):
        return {"online": jax.device_get(self.online), "target": jax.device_get(self.target),
                "opt_state": jax.device_get(self.opt_state), "epsilon": self.epsilon,
                "total_steps": self.total_steps, "rng": self.rng.bit_generator.state,
                "buffer": self.buffer.state_dict()}

    def load_state_dict(self, state):
        self.online = jax.tree.map(jnp.asarray, state["online"])
        self.target = jax.tree.map(jnp.asarray, state["target"])
        self.opt_state = jax.tree.map(jnp.asarray, state["opt_state"])
        self.epsilon, self.total_steps = state["epsilon"], state["total_steps"]
        self.rng.bit_generator.state = state["rng"]; self.buffer.load_state_dict(state["buffer"])


## 7. Persistence, logging, and resume


In [ ]:
import csv, os, pickle, time
from datetime import datetime, timezone
FIELDS = ["timestamp", "episode", "total_steps", "reward", "loss", "epsilon", "episode_length", "steps_per_second", "success"]

def save_state(agent, episode):
    payload = {"version": 1, "episode": episode, "agent": agent.state_dict()}
    path = CHECKPOINT_DIR / f"checkpoint_{agent.total_steps:012d}.pkl"; temp = path.with_suffix(".tmp")
    with temp.open("wb") as file:
        pickle.dump(payload, file, pickle.HIGHEST_PROTOCOL); file.flush(); os.fsync(file.fileno())
    temp.replace(path)
    for old in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"))[:-3]: old.unlink()
    return path

def restore_state(agent):
    for path in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"), reverse=True):
        try:
            with path.open("rb") as file: state = pickle.load(file)
            if state["version"] != 1: raise ValueError("version")
            agent.load_state_dict(state["agent"]); print("Resumed", path.name); return state["episode"]
        except Exception as error: print("Skipped", path.name, error)
    return 0

def log_row(row):
    new = not METRICS_PATH.exists()
    with METRICS_PATH.open("a", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=FIELDS)
        if new: writer.writeheader()
        writer.writerow(row); file.flush(); os.fsync(file.fileno())


## 8. Train
The first update includes JAX compilation and is slower than later updates.


In [ ]:
env = SinglePendulumEnv(True, MAX_EPISODE_STEPS, ACTION_LIMIT)
agent = DQNAgent(env.observation_space.shape[0], env.action_space.n, SEED)
start_episode = restore_state(agent); clock = time.perf_counter(); start_steps = agent.total_steps
try:
    episode = start_episode
    while agent.total_steps < TOTAL_STEPS:
        episode += 1; obs, _ = env.reset(seed=SEED + episode)
        reward_sum, losses, success = 0.0, [], False
        for length in range(1, MAX_EPISODE_STEPS + 1):
            action = agent.act(obs); next_obs, reward, terminated, truncated, info = env.step(action)
            loss = agent.observe(obs, action, reward, next_obs, terminated)
            if loss is not None: losses.append(loss)
            obs = next_obs; reward_sum += reward; success |= abs(info["theta"]) < .35
            if terminated or truncated or agent.total_steps >= TOTAL_STEPS: break
        elapsed = max(time.perf_counter()-clock, 1e-9)
        log_row({"timestamp": datetime.now(timezone.utc).isoformat(), "episode": episode,
                 "total_steps": agent.total_steps, "reward": reward_sum,
                 "loss": np.mean(losses) if losses else np.nan, "epsilon": agent.epsilon,
                 "episode_length": length, "steps_per_second": (agent.total_steps-start_steps)/elapsed,
                 "success": int(success)})
        if episode % CHECKPOINT_EVERY == 0: print("Saved", save_state(agent, episode))
        if episode % 10 == 0: print(episode, agent.total_steps, round(reward_sum, 1), round(agent.epsilon, 3))
finally:
    save_state(agent, episode,)


## 9. Static dashboard


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.read_csv(METRICS_PATH).drop_duplicates("episode", keep="last").sort_values("episode")
window = min(20, len(metrics))
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
fig.suptitle(TITLE, fontsize=16, fontweight="bold")
axes[0, 0].plot(metrics.episode, metrics.reward, alpha=.3)
axes[0, 0].plot(metrics.episode, metrics.reward.rolling(window, min_periods=1).mean())
axes[0, 0].set_title("Reward and moving average")
axes[0, 1].plot(metrics.episode, metrics.loss); axes[0, 1].set_title("Training loss")
axes[0, 2].plot(metrics.episode, metrics.epsilon); axes[0, 2].set_title("Epsilon")
axes[1, 0].plot(metrics.episode, metrics.episode_length); axes[1, 0].set_title("Episode length")
axes[1, 1].plot(metrics.episode, metrics.steps_per_second); axes[1, 1].set_title("Steps/second")
axes[1, 2].plot(metrics.episode, metrics.success); axes[1, 2].set_title("Upright reached")
for axis in axes.flat:
    axis.set_xlabel("Episode"); axis.grid(alpha=.25)
fig.savefig(DASHBOARD_PATH, dpi=160)
plt.show()
print("Saved", DASHBOARD_PATH)


## 10. Greedy evaluation


In [ ]:
rewards = []
for episode in range(10):
    obs, _ = env.reset(seed=20_000 + episode); total = 0.0
    while True:
        obs, reward, terminated, truncated, info = env.step(agent.act(obs, greedy=True)); total += reward
        if terminated or truncated: break
    rewards.append(total)
print("Greedy reward mean/std:", np.mean(rewards), np.std(rewards))


## Replay The Trained Run

Colab cannot reliably open MuJoCo's interactive desktop viewer. This block runs a
deterministic evaluation, streams rendered frames directly into an MP4, saves it in
the run directory, and displays it inline. It replays the current trained policy or
controller; it is not an exact recording of a stochastic training episode.


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

REPLAY_SEED, REPLAY_SECONDS, REPLAY_FPS = 50_002, 10.0, 50
REPLAY_PATH = OUTPUT_DIR / "replay.mp4"
replay_env = SinglePendulumEnv(True, int(REPLAY_SECONDS / 0.005), ACTION_LIMIT)
observation, _ = replay_env.reset(seed=REPLAY_SEED)
renderer = mujoco.Renderer(replay_env.model, height=480, width=640)
writer = imageio.get_writer(REPLAY_PATH, fps=REPLAY_FPS, codec="libx264", quality=8)
frame_stride = max(1, round(1 / (replay_env.dt * REPLAY_FPS)))
try:
    for step in range(replay_env.max_episode_steps):
        observation, _, terminated, truncated, _ = replay_env.step(agent.act(observation, greedy=True))
        if step % frame_stride == 0:
            renderer.update_scene(replay_env.data, camera="replay")
            writer.append_data(renderer.render())
        if terminated or truncated:
            break
finally:
    writer.close()
    renderer.close()
print("Saved replay:", REPLAY_PATH)
display(Video(str(REPLAY_PATH), embed=True))
